# n2c2 2018 Dataset Analysis

Exploratory data analysis of the n2c2 2018 clinical trial eligibility dataset.

**Note**: Requires n2c2 data access from Harvard DBMI Portal.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from trial_matcher.config import get_settings
from trial_matcher.evaluation.n2c2_loader import N2C2Loader, N2C2_CRITERIA

settings = get_settings()

In [ ]:
# Load n2c2 data
loader = N2C2Loader(settings.n2c2_data_dir)
dataset = loader.load()
print(f'Loaded {dataset.n_patients} patients, {dataset.n_criteria} criteria')
print(f'Total annotated pairs: {dataset.n_pairs}')

In [ ]:
# Prevalence per criterion
rows = []
for patient in dataset.patients:
    for criterion, label in patient.labels.items():
        rows.append({'patient_id': patient.patient_id, 'criterion': criterion, 'label': label})

df = pd.DataFrame(rows)
labeled = df[df.label != -1]

prevalence = labeled.groupby('criterion')['label'].agg(['mean', 'count']).reset_index()
prevalence.columns = ['criterion', 'prevalence', 'n']
prevalence = prevalence.sort_values('prevalence', ascending=False)

print('Criterion Prevalence (fraction of patients meeting criteria):')
print(prevalence.to_string(index=False))

In [ ]:
# Plot prevalence
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(prevalence['criterion'], prevalence['prevalence'], color='steelblue')
ax.axhline(0.5, color='red', linestyle='--', alpha=0.5, label='50% threshold')
ax.set_ylabel('Prevalence (fraction met)')
ax.set_title('n2c2 2018 Criterion Prevalence')
ax.set_xticklabels(prevalence['criterion'], rotation=45, ha='right')
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('../results/n2c2_prevalence.png', dpi=150, bbox_inches='tight')
plt.show()